In [ ]:
#Import necessary Liabraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report,
                             roc_curve)

RANDOM_STATE = 42
DATA_PATH = "Churn_Modelling.csv"


## 1. Load the dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
print("\nTarget distribution:")
display(df["Exited"].value_counts())


## 2. Feature engineering

In [ ]:
def add_engineered_features(data):
    data = data.copy()
    data["BalancePerProduct"] = (
        data["Balance"] / data["NumOfProducts"].replace(0, np.nan)
    ).fillna(0)

    data["AgeBand"] = pd.cut(
        data["Age"], bins=[17, 29, 39, 49, 59, np.inf],
        labels=["18-29", "30-39", "40-49", "50-59", "60+"]
    )
    data["TenureBand"] = pd.cut(
        data["Tenure"], bins=[-1, 2, 5, 8, 10],
        labels=["0-2", "3-5", "6-8", "9-10"]
    )
    data["CreditScoreBand"] = pd.cut(
        data["CreditScore"], bins=[-np.inf, 579, 669, 739, 799, np.inf],
        labels=["<580", "580-669", "670-739", "740-799", "800+"]
    )
    data["IsBalanceZero"] = (data["Balance"] == 0).astype(int)
    return data

df = add_engineered_features(df)

X = df.drop(columns=["RowNumber", "CustomerId", "Surname", "Exited"])
y = df["Exited"]

print("Features:", X.shape)


## 3. Stratified train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))


## 4. Preprocessing pipeline

In [ ]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])


## 5. Fit Random Forest

The model configuration matches the Part B specification.

In [ ]:
model = RandomForestClassifier(n_estimators=300, max_depth=12,
                              min_samples_split=5, min_samples_leaf=2,
                              class_weight="balanced", random_state=42, n_jobs=-1)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print("Random Forest fitted successfully.")


## 6. Performance metrics

In [ ]:
results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"],
    "Score": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0),
        roc_auc_score(y_test, y_prob)
    ]
})

results["Percentage"] = results["Score"] * 100
display(results.style.format({"Score": "{:.4f}", "Percentage": "{:.2f}%"}))


## 7. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not Churned", "Churned"]
)
disp.plot()
plt.title("Model 2 – Random Forest – Confusion Matrix")
plt.show()


## 8. Classification report

In [ ]:
print(classification_report(
    y_test, y_pred,
    target_names=["Not Churned", "Churned"],
    zero_division=0
))


## 9. ROC curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_value = roc_auc_score(y_test, y_prob)

plt.figure()
plt.plot(fpr, tpr, label=f"Random Forest (AUC = {auc_value:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model 2 – Random Forest – ROC Curve")
plt.legend()
plt.show()


## 10. Save the performance results

In [ ]:
results.to_csv("Model2Performance_results.csv", index=False)
print("Saved:", "Model2Performance_results.csv")
